# API 4-2: REST APIs 

Non-GET, Caching Strategies, Azure 

## HTTP Methods beyond GET

So far we have only seen GET requests. But there are other HTTP methods that are used to perform different operations  such as:

- POST: Create a new resource
- PUT: Update an existing resource
- DELETE: Delete a resource

From the perspective of `requests` these behave the same as a GET. The only subtle difference is that you can pass a `data` parameter when there are large payloads which cannot fit on the URL.  


## Example of a POST request

Let's do an example with the Azure sentiment API from the IoT Portal. This REST API requires the text to analyze, and will return the sentiment or "mood" of the text.  Since there can be a substantial amount of text, the API requires the POST method, and the data to be sent in the body of the request.

First, Try it out in the IoT portal:

`/api/azure/sentiment`

Text: "I love IST356. It is the best course I've ever taken."

Then, let's convert the `curl` command to Python `requests` code. 

In [ ]:
import requests

'''
curl -X 'POST' \
  'https://cent.ischool-iot.net/api/azure/sentiment' \
  -H 'accept: application/json' \
  -H 'X-API-KEY: APIKEY' \
  -H 'Content-Type: application/x-www-form-urlencoded' \
  -d 'text=I%20love%20IST356.%20It%20is%20the%20best%20course%20I'\''ve%20ever%20taken.'
'''

apikey = '632985c9646a0bc8f547f1d9'
url = 'https://cent.ischool-iot.net/api/azure/sentiment'
headers = {
    'accept': 'application/json',
    'X-API-KEY': apikey,
    'Content-Type': 'application/x-www-form-urlencoded'
}
data = {
    'text': 'I love IST356. It is the best course I\'ve ever taken.'
}
response = requests.post(url, headers=headers, data=data)
response.raise_for_status()
results = response.json()
sentiment = results['results']['documents'][0]['sentiment']
print(sentiment)

positive


In [6]:
import pandas as pd


entities=[   
          {
            "text": "pizza",
            "category": "Product",
            "offset": 4,
            "length": 5,
            "confidenceScore": 1
          },
          {
            "text": "store",
            "category": "Location",
            "offset": 40,
            "length": 5,
            "confidenceScore": 0.85
          }]

df = pd.DataFrame(entities)
df

,text,category,offset,length,confidenceScore
0,pizza,Product,4,5,1.00
1,store,Location,40,5,0.85


## Challenge 4-2-1

For this challenge, use Azure entity recognition API to extract entities from the following text.

"The Dallas Cowboys are a far better team than the New York Giants this year. The Giants have not won a conference game yet."

Using the API output, print each extracted entity and its type.

Steps 
1. Create a function that takes a string and returns a dict (response from API)
    1. Include API endpoint for entity recognition 
    2. text to be sent to the API
    3. headers which include the API key for authentication 
 2. Make API request 
    1. Send text to API
    2. Convert API response to a Python dictionary
 3. Streamlit input
    1. Create a text box in the app
 4. Create a button that runs the code when the user clicks the button 
 5. Check if text exist. Prevent empty input from being processed.
 6. Extract entities from JSON
 7. Convert to a dataframe
 8. Display results

## Caching Strategies

When you are making a lot of requests to an API, it is a good idea to cache the results. We don't want to make the same request over and over again if we don't have to as this can effect our rate limits and thus our pricing.

Caching can be done in a number of ways. The simplest method is a Python dictionary where the key is the request and the value is the response. You can use Python `pickle` library to save the dictionary to disk so that you can use it across multiple sessions.

Pickle in Python is a built-in module used for serializing and deserializing objects.

What that means
Serialization (pickling) → converting a Python object into a byte stream (so it can be saved to a file or sent over a network)
Deserialization (unpickling) → converting that byte stream back into a Python object

Why use pickle?

It lets you:

- Save complex Python objects (lists, dictionaries, models, etc.)
- Reload them later exactly as they were
- Share objects between programs



The caching strategy looks like this:

    1. Check if the request is in the cache
    2. If it is, we have a CACHE hit:
    3.    return the response from the cache
    4. else (cache miss)
    5.    Call the API to get the response
    6.    Add response to cache

Let's do an example with the Google Geocoding API on the IoT Portal. 


In [1]:
import pickle

data = {"name": "Angela", "score": 95}

with open("dataexample.pkl", "wb") as file:
    pickle.dump(data, file)

In [2]:
import pickle

with open("dataexample.pkl", "rb") as file:
    loaded_data = pickle.load(file)

print(loaded_data)

{'name': 'Angela', 'score': 95}


In [14]:
cache={
    'Mount Rushmore': {
        'lat': 43.8791,
        'lng': -103.4591
    },
    'JMA Building, Syracuse University': {
        'lat': 43.0379,
        'lng': -76.1342
    },
    'Varsity Pizza': {
        'lat': 43.0375,
        'lng': -76.1320
    }
}


In [ ]:
cache ={"Hinds Hall, Syracuse University": {'lat': 43.0376, 'lng': -76.1345}}

In [3]:
import requests
import requests_cache as rq
import pickle as pk1

apikey = '632985c9646a0bc8f547f1d9'
pickle_file = 'geocode_cache.pkl'
location = 'Syracuse, NY'
cache_key = location.lower() # create cache key by converting location to lowercase
cache = rq.clear_cache(pickle_file) # load cache from file, if it exists, otherwise create empty cache
for i in range(3): # loop to test cache by by making the same request multiple times
    # check if in cache
    if cache_key in cache.keys(): # check if location in cache
        print('Using cache')
        geo = cache[cache_key] # if found retrieve from cache

    else:
        # not in cache, make request
        print('Making request')
        url = 'https://cent.ischool-iot.net/api/google/geocode'
        headers = { 'X-API-KEY': apikey }
        params = {'location': location}
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status()
        geo = response.json()
        #save it in the cache
        cache[cache_key] = geo
        rq.save_cache(cache, pickle_file)

    print(geo['results'][0]['geometry']['location'])



Making request
{'lat': 43.0494832, 'lng': -76.14739770000001}
Using cache
{'lat': 43.0494832, 'lng': -76.14739770000001}
Using cache
{'lat': 43.0494832, 'lng': -76.14739770000001}


## Challenge 4-2-2

Come up with a caching strategy for the Azure sentiment API. Use the `requests_cache.py` module. You will need to decided on the cache key.

For the following input:

for each text in the list:  
output the text, the sentiment, and if the results came from the API or cache.

```
texts = [
    "I love IST356. It is the best course I've ever taken.", 
    "I hate the New York Giants.",
    "I love IST356. It is the best course I've ever taken.", 
    "I don't like the New York Giants."
]
```

Steps 

1. Initialize cache
2. API setup
3. Loop through text data
4. Check cache first
   1. If text already analyzed get use stored result. No API call
   2. If text NOT in cache. Send to API. Get sentiment result.
5. Save result to cache
6. Label source of data e.g cached or not cached
7. Extract sentiment from JSON structure
8. Output sentiment

Workflow 

Check cache
If found → use it
If not → call API
Save result
Reuse later